---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---

# Machine Learning Algorithms Comparison

**Objective:** Compare the performance of 4 different algorithms for sonic log prediction.

**Algorithms compared:**
1. XGBoost
2. LightGBM
3. HistGradientBoosting
4. RandomForest

**Methodology:**
- All used GroupKFold (cv=10) for optimization
- All used LOWO for final validation
- Same features: GR, RT90, RHOB, NPHI
- Same target: DT

**Analyses:**
1. Metrics comparison (R², RMSE, MAE)
2. Performance distribution
3. Problematic wells
4. Computational time
5. Best algorithm identification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Load results from all algorithms 

RESULTS_DIR = Path('../results')

algorithms = {
    'XGBoost': RESULTS_DIR / 'xgboost',
    'LightGBM': RESULTS_DIR / 'lightgbm',
    'HistGradientBoosting': RESULTS_DIR / 'histgb',
    'RandomForest': RESULTS_DIR / 'randomforest'
}

results = {}
predictions = {}

print("LOADING RESULTS")

for algo_name, path in algorithms.items():
    if algo_name == 'XGBoost':
        metrics_file = path / 'metrics' / 'lowo_xgboost_results.csv'
        pred_file = path / 'predictions' / 'lowo_xgboost_predictions.csv'
    elif algo_name == 'LightGBM':
        metrics_file = path / 'metrics' / 'lowo_lightgbm_results.csv'
        pred_file = path / 'predictions' / 'lowo_lightgbm_predictions.csv'
    elif algo_name == 'HistGradientBoosting':
        metrics_file = path / 'metrics' / 'lowo_histgb_results.csv'
        pred_file = path / 'predictions' / 'lowo_histgb_predictions.csv'
    else:
        metrics_file = path / 'metrics' / 'lowo_rf_results.csv'
        pred_file = path / 'predictions' / 'lowo_rf_predictions.csv'
    
    if metrics_file.exists():
        results[algo_name] = pd.read_csv(metrics_file)
        print(f"{algo_name:25s}: {len(results[algo_name])} wells loaded")
    else:
        print(f"{algo_name:25s}: File not found at {metrics_file}")
    
    if pred_file.exists():
        predictions[algo_name] = pd.read_csv(pred_file)

print(f"\nAvailable algorithms: {len(results)}")

In [ ]:
# General comparison table
print("OVERALL PERFORMANCE COMPARISON")

comparison_data = []

for algo_name, df in results.items():
    comparison_data.append({
        'Algoritmo': algo_name,
        'R²_mean': df['R2'].mean(),
        'R²_median': df['R2'].median(),
        'R²_std': df['R2'].std(),
        'R²_min': df['R2'].min(),
        'R²_max': df['R2'].max(),
        'RMSE_mean': df['RMSE'].mean(),
        'MAE_mean': df['MAE'].mean(),
        'N_wells': len(df),
        'N_problematic': len(df[df['R2'] < 0.70])
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('R²_mean', ascending=False)

print("\nStatistical Summary:\n")
print(comparison_df.to_string(index=False))

# Save
comparison_df.to_csv('../results/comparison/metrics/algorithms_comparison.csv', index=False)
print("\nTable saved: ../results/comparison/metrics/algorithms_comparison.csv")

In [ ]:
# Algorithm Ranking
print("RANKING BY CRITERION")

print("\nRanking by mean R²:")
for i, row in comparison_df.iterrows():
    rank = comparison_df.index.get_loc(i) + 1
    print(f"    {rank}. {row['Algoritmo']:25s}: R² = {row['R²_mean']:.4f}")

print("\nLowest RMSE (best):")
best_rmse = comparison_df.nsmallest(1, 'RMSE_mean')
print(f"   {best_rmse.iloc[0]['Algoritmo']}: RMSE = {best_rmse.iloc[0]['RMSE_mean']:.3f}")

print("\nFewest problematic wells (R² < 0.50):")
best_prob = comparison_df.nsmallest(1, 'N_problematic')
print(f"   {best_prob.iloc[0]['Algoritmo']}: {int(best_prob.iloc[0]['N_problematic'])} wells")

print("\nLowest variability (smallest std):")
best_std = comparison_df.nsmallest(1, 'R²_std')
print(f"   {best_std.iloc[0]['Algoritmo']}: std = {best_std.iloc[0]['R²_std']:.4f}")

In [ ]:
# Visualization: Comparative Boxplots
print("GENERATING VISUALIZATIONS")

plot_data = []
for algo_name, df in results.items():
    for r2 in df['R2']:
        plot_data.append({'Algoritmo': algo_name, 'R²': r2})

plot_df = pd.DataFrame(plot_data)

fig, ax = plt.subplots(figsize=(12, 7))

colors = {'XGBoost': 'steelblue', 'LightGBM': 'orange', 
          'HistGradientBoosting': 'green', 'RandomForest': 'red'}

bp = ax.boxplot(
    [results[algo]['R2'] for algo in comparison_df['Algoritmo']],
    labels=comparison_df['Algoritmo'],
    patch_artist=True,
    showmeans=True,
    meanprops=dict(marker='D', markerfacecolor='yellow', markersize=8)
)

for patch, algo in zip(bp['boxes'], comparison_df['Algoritmo']):
    patch.set_facecolor(colors.get(algo, 'lightblue'))
    patch.set_alpha(0.7)

ax.axhline(y=0.90, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Excellent (0.90)')
ax.axhline(y=0.80, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='Good (0.80)')
ax.axhline(y=0.50, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Problematic (0.50)')

ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_xlabel('Algorithm', fontsize=12, fontweight='bold')
ax.set_title('Performance Comparison — R² by Algorithm', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/comparison/figures/algorithms_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

print('Boxplot saved: ../results/comparison/figures/algorithms_boxplot.png')

In [ ]:
# Chart: R² per Well — Comparison Across Tree-Based Algorithms

ALGO_COLORS = {
    'LightGBM'           : '#2ECC71',
    'XGBoost'            : '#E74C3C',
    'HistGradientBoosting': '#3498DB',
    'RandomForest'       : '#F39C12',
}

r2_per_well = {}
for algo, df in results.items():
    r2_per_well[algo] = df.set_index('Well_Name')['R2']

df_r2 = pd.DataFrame(r2_per_well)

sort_col = 'LightGBM' if 'LightGBM' in df_r2.columns else df_r2.columns[0]
df_r2 = df_r2.sort_values(sort_col, ascending=True)

algos   = list(df_r2.columns)
n_algos = len(algos)
n_wells = len(df_r2)
bar_h   = 0.8 / n_algos

fig, ax = plt.subplots(figsize=(12, max(10, n_wells * 0.45)))

y_positions = np.arange(n_wells)

for i, algo in enumerate(algos):
    offset = (i - n_algos / 2 + 0.5) * bar_h
    vals   = df_r2[algo].values
    color  = ALGO_COLORS.get(algo, '#95A5A6')

    ax.barh(y_positions + offset, vals,
            height=bar_h * 0.9,
            color=color, alpha=0.85,
            edgecolor='white', linewidth=0.4,
            label=algo)

r2_mean = df_r2[sort_col].mean()
ax.axvline(r2_mean, color='black', linestyle='--', linewidth=1.2,
           label=f'Mean R² {sort_col} = {r2_mean:.4f}')

ax.axvline(0.70, color='purple', linestyle='--', linewidth=1.5,
           label='Threshold R² = 0.70')

ax.set_yticks(y_positions)
ax.set_yticklabels([w.replace('-SES', '') for w in df_r2.index], fontsize=8)
ax.set_xlabel('R² LOWO', fontsize=12)
ax.set_title('R² Comparison per Well — Tree-Based Algorithms\n'
             'Sergipe-Alagoas Basin | Leave-One-Well-Out Validation',
             fontsize=13)

ax.set_xlim(0, 1.08)
ax.set_xticks(np.arange(0, 1.1, 0.1))
ax.grid(alpha=0.25, axis='x', linestyle='-', linewidth=0.8)

ax.legend(loc='lower right', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.savefig('../results/comparison/figures/r2_por_poco_algoritmos.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f'Chart saved: ../results/comparison/figures/r2_por_poco_algoritmos.png')
print(f'Wells plotted: {n_wells} | Algorithms: {algos}')

In [ ]:
# Well-by-Well Comparison
print("WELL-BY-WELL ANALYSIS")

best_algo = comparison_df.iloc[0]['Algoritmo']
best_r2 = results[best_algo].set_index('Well_Name')['R2']

print(f"\nBest algorithm: {best_algo}")
print(f"   Mean R²: {best_r2.mean():.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, algo_name in enumerate([a for a in comparison_df['Algoritmo'] if a != best_algo]):
    ax = axes[idx]
    
    other_r2 = results[algo_name].set_index('Well_Name')['R2']
    
    common_wells = best_r2.index.intersection(other_r2.index)
    x = best_r2[common_wells]
    y = other_r2[common_wells]
    
    ax.scatter(x, y, alpha=0.6, s=80, edgecolors='black', linewidth=0.5)
    
    lims = [0, 1]
    ax.plot(lims, lims, 'r--', alpha=0.5, linewidth=2, label='y=x (equal)')
    
    diff = (y - x).mean()
    color = 'green' if diff > 0 else 'red'
    
    ax.set_xlabel(f'{best_algo} R²', fontsize=11, fontweight='bold')
    ax.set_ylabel(f'{algo_name} R²', fontsize=11, fontweight='bold')
    ax.set_title(f'{best_algo} vs {algo_name}\nMean ΔR² = {diff:+.4f}', 
                 fontsize=12, fontweight='bold', color=color)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../results/comparison/figures/algorithms_scatter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(' Scatter plots saved: ../results/comparison/figures/algorithms_scatter_comparison.png')

In [ ]:
# Problematic Wells Analysis
print("PROBLEMATIC WELLS BY ALGORITHM")

threshold = 0.70

for algo_name, df in results.items():
    problematic = df[df['R2'] < threshold]
    
    print(f"\n{algo_name}:")
    print(f"   Problematic wells: {len(problematic)} ({len(problematic)/len(df)*100:.1f}%)")
    
    if len(problematic) > 0:
        print(f"   List:")
        for _, row in problematic.iterrows():
            print(f"      - {row['Well_Name']:25s} | R² = {row['R2']:.4f}")

print('\n')
print("WELLS PROBLEMATIC ACROSS ALL ALGORITHMS")

problematic_sets = []
for algo_name, df in results.items():
    prob_wells = set(df[df['R2'] < threshold]['Well_Name'])
    problematic_sets.append(prob_wells)

consistent_problems = set.intersection(*problematic_sets) if problematic_sets else set()

if consistent_problems:
    print(f"\n  {len(consistent_problems)} wells are problematic in ALL algorithms:")
    for well in sorted(consistent_problems):
        print(f"   - {well}")
        for algo_name, df in results.items():
            r2 = df[df['Well_Name'] == well]['R2'].values[0]
            print(f"      {algo_name:25s}: R² = {r2:.4f}")
else:
    print("\n No well is problematic in all algorithms!")

In [ ]:
# FInal Recommendations
print("RECOMMENDATIONS")

best_algo = comparison_df.iloc[0]['Algoritmo']
best_r2 = comparison_df.iloc[0]['R²_mean']
best_rmse = comparison_df.iloc[0]['RMSE_mean']

print(f"\n RECOMMENDED ALGORITHM: {best_algo}")
print(f"\n Performance:")
print(f"   Mean R²:  {best_r2:.4f}")
print(f"   Mean RMSE: {best_rmse:.3f} μs/ft")
print(f"   Problematic wells: {int(comparison_df.iloc[0]['N_problematic'])}")
